# Phase 5: Diagnosing Outputs from Running XML-sets and Generate Report

This notebook  attempts to do what the software [Tracer](https://beast.community/tracer) does and make some improvements. 

## Instructions

Code cells of this Jupyter notebook should be run sequentially via shift+enter. Several cells will produce widgets that allow you to make various selections to select for MCMC chains that have converged. Once you have made that selection left-click on the cell below and press shift+enter.

## Suggested Reading

Up to and including "**x% HPD interval**" of:

Drummond, Alexei J., and Bouckaert, Remco R. ‘Ch 10: Posterior Analysis and Post Processing.’ In Bayesian Evolutionary Analyses with BEAST. Cambridge University Press, 2015. https://www.cambridge.org/core/books/bayesian-evolutionary-analysis-with-beast/81F5894F05E87F13C688ADB00178EE00.

The authors have been kind enough to make a draft copy of the book available at http://alexeidrummond.org/assets/publications/2015-drummond-bayesian.pdf.

## Parameters
<details>
    <summary>Click To See A Decription of Parameters</summary>
        <pre>
            <code>
beast_outputs: str
    A valid path containing output `.log` and `.tree` files from running BEAST 2

parameters_report_template: str
    Name of a valid report template to use to generate report on parameters.

kernel: str, default 'beast_pype'
    Name of Jupyter python kernel to use when running report template notebooks.

topology : str
    Topology to use for merged BEAST tree summarization and plotting, e.g. "MCC" or "CCD0". 
    See BEAST2's TreeAnnotator documentation or https://www.beast2.org/2024/06/24/what-is-new-in-v2.7.7.html for more details on tree summarization methods.

summary_tree_low_memory: bool, defaults to False
    Whether to use low memory option when using BEAST 2's TreeAnnotator to summarize merged BEAST trees. Doing so will take more time. See BEAST2's TreeAnnotator documentation.

collection_date_field: str
    Name of field in metadata_db containing collection dates of sequences. Should be formatted YYYY-MM-DD.

  </code>
</pre>

In [ ]:
parameters_report_template = None
xml_set_label = 'xml_set'
beast_xml_path = None
kernel_name = 'beast_pype'
topology = "CCD0"
summary_tree_low_memory = False
collection_date_field = "collection date"

Import necessary packages.

In [ ]:
from copy import deepcopy
import yaml
import papermill as pm
from beast_pype.nb_utils import execute_notebook
import pandas as pd
from beast_pype.diagnostics.mcmc import BEASTDiag
from beast_pype.report_gen import add_unreported_outputs, gen_summary_tree_notebook, gen_tree_report, gen_metadata_report
from beast_pype.diagnostics.runtime import get_beast_runtimes, get_slurm_job_stats
import pandas as pd
import os
from beast_pype.path_utils import path_to_workflow_modules, path_to_report_templates
import warnings
warnings.filterwarnings( "ignore") # Stop annoying warnings.

In [ ]:
workflow_modules_path = path_to_workflow_modules()
report_modules_path = path_to_report_templates()
parameters_report_template = f"{report_modules_path}/{parameters_report_template}.ipynb"
save_dir=os.getcwd()

Create outputs_and_reports directory

In [ ]:
outputs_and_reports_dir = f'{save_dir}/outputs_and_reports'
if not os.path.exists(outputs_and_reports_dir):
    os.makedirs(outputs_and_reports_dir)
else:
    print(f'{outputs_and_reports_dir} already exists. If you proceed you overwrite files this directory.')

Set up dictionary of beast_outputs_paths for xml_sets.

In [ ]:
beast_outputs_paths = {}